# Feature Workflow: Baseline -> Enhanced -> Diagnostics -> Reduced

This notebook follows the exact phase order:
1. List current features (baseline + engineered) and motivate each feature.
2. Train/evaluate baseline feature set.
3. Train/evaluate enhanced feature set.
4. Run feature diagnostics (permutation drop / SHAP / goodness outputs from pipeline).
5. Retrain on reduced feature set and compare.

DuckDB is explicitly used for the JSON -> many-to-many normalization step.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
import duckdb

PROJECT_ROOT = Path('/home/djameldino/Big-Data/big_data_assignment')
BASE = PROJECT_ROOT / 'members' / 'djamel' / 'outputs_restart'
FIG_DIR = BASE / 'notebook_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

YELLOW = '#F5C518'
BLUE = '#1848f5'
BLACK = '#000000'
CARD = '#111111'
WHITE = '#FFFFFF'

print('BASE:', BASE)
print('FIG_DIR:', FIG_DIR)


ValueError: All ufuncs must have type `numpy.ufunc`. Received (<ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>)

## 0) Explicit DuckDB check for JSON -> many-to-many

In [ ]:
raw_json_dir = PROJECT_ROOT / 'data' / 'raw' / 'json'
directing_json = raw_json_dir / 'directing.json'
writing_json = raw_json_dir / 'writing.json'

con = duckdb.connect()
directing_edges = con.execute(
    """
    WITH src AS (
      SELECT json(movie) AS movie_obj, json(director) AS director_obj
      FROM read_json_auto(?)
    ),
    movies AS (
      SELECT je.key AS k, trim(both '\"' from CAST(je.value AS VARCHAR)) AS tconst
      FROM src, json_each(movie_obj) je
    ),
    directors AS (
      SELECT je.key AS k, trim(both '\"' from CAST(je.value AS VARCHAR)) AS director_id
      FROM src, json_each(director_obj) je
    )
    SELECT COUNT(*)
    FROM movies m
    JOIN directors d USING (k)
    """,
    [str(directing_json)],
).fetchone()[0]

writing_edges = con.execute(
    "SELECT COUNT(*) FROM read_json_auto(?)",
    [str(writing_json)],
).fetchone()[0]

print('DuckDB directing edges:', directing_edges)
print('DuckDB writing edges  :', writing_edges)


## 1) Load features and list baseline vs engineered sets

In [ ]:
train = pd.read_csv(BASE / 'train_features.csv')
val = pd.read_csv(BASE / 'validation_features.csv')

all_features = [c for c in train.columns if c not in ['tconst', 'label']]
baseline_features = [
    'startYear',
    'endYear',
    'runtimeMinutes_capped',
    'numVotes_log1p_capped',
    'title_len',
    'has_original_title',
]
baseline_features = [f for f in baseline_features if f in all_features]
engineered_features = [f for f in all_features if f not in baseline_features]

print('Baseline features:', len(baseline_features))
print(baseline_features)
print('\nEngineered features:', len(engineered_features))
print(engineered_features)
print('\nTotal model features:', len(all_features))


## 2) Feature motivation table (every feature)

In [ ]:
feature_motivation = {
    'startYear': 'Release era signal; film markets and rating behavior change over time.',
    'endYear': 'Series/end timing signal; helps separate title lifecycle patterns.',
    'runtimeMinutes_capped': 'Runtime effect with outlier control to avoid extreme-value distortion.',
    'numVotes_log1p_capped': 'Popularity proxy; log+capping handles heavy-tailed vote counts.',
    'title_len': 'Simple lexical complexity proxy for title style.',
    'title_word_count': 'Title structure complexity signal.',
    'title_has_digit': 'Franchise/sequel/year marker signal in titles.',
    'title_has_colon': 'Subtitle/franchise formatting signal.',
    'title_has_question': 'Title style marker potentially linked to genre/tone.',
    'title_upper_ratio': 'Typography/style marker for naming conventions.',
    'has_original_title': 'Localization/remake/translation proxy.',
    'runtime_missing': 'Missingness can itself be informative.',
    'votes_missing': 'Missingness can itself be informative.',
    'start_missing': 'Missingness can itself be informative.',
    'end_missing': 'Missingness can itself be informative.',
    'year_span': 'Duration/lifecycle feature when both years are present.',
    'num_directors': 'Team size effect from many-to-many credits.',
    'num_unique_directors': 'Director diversity effect.',
    'num_writers': 'Writing team size effect.',
    'num_unique_writers': 'Writer diversity effect.',
    'is_auteur': 'Single-director/single-writer concentration proxy.',
    'director_hit_rate': 'Leak-safe OOF target encoding of director history.',
    'writer_hit_rate': 'Leak-safe OOF target encoding of writer history.',
    'canonical_title_hit_rate': 'Leak-safe OOF prior success by normalized title.',
    'title_group_size_train': 'How often canonical title appears in training.',
    'title_unique_years_train': 'Title ambiguity/remake proxy (same title across years).',
    'title_conflicting_years': 'Binary conflict flag for canonical-title year mismatch.',
    'title_sim_to_hit': 'Cosine similarity of title TF-IDF to hit centroid.',
    'title_sim_to_non_hit': 'Cosine similarity of title TF-IDF to non-hit centroid.',
    'title_sim_margin': 'Net semantic tilt toward hit-like vs non-hit-like title language.',
}

motivation_rows = []
for f in all_features:
    motivation_rows.append({
        'feature': f,
        'feature_group': 'baseline' if f in baseline_features else 'engineered',
        'motivation': feature_motivation.get(f, 'MOTIVATION_MISSING'),
    })
motivation_df = pd.DataFrame(motivation_rows)
missing_motivation = motivation_df[motivation_df['motivation'] == 'MOTIVATION_MISSING']

display(motivation_df)
if len(missing_motivation):
    print('Missing motivation for:', missing_motivation['feature'].tolist())
else:
    print('All features have explicit motivation.')

motivation_df.to_csv(BASE / 'feature_motivation_table.csv', index=False)


## 3) Train/evaluate helper

In [ ]:
def evaluate_models(train_df, val_df, features, stage_name):
    y_train = train_df['label'].astype(int)
    y_val = val_df['label'].astype(int)
    med = train_df[features].median(numeric_only=True)
    X_train = train_df[features].fillna(med)
    X_val = val_df[features].fillna(med)

    rows = []

    # Logistic
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    log_model = LogisticRegression(max_iter=2000, random_state=42)
    log_model.fit(X_train_s, y_train)
    p_log = log_model.predict_proba(X_val_s)[:, 1]
    pred_log = (p_log >= 0.5).astype(int)
    rows.append({
        'stage': stage_name,
        'model': 'logistic',
        'feature_count': len(features),
        'accuracy': float(accuracy_score(y_val, pred_log)),
        'roc_auc': float(roc_auc_score(y_val, p_log)),
    })

    # XGBoost
    xgb_model = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        random_state=42,
        n_jobs=4,
    )
    xgb_model.fit(X_train, y_train)
    p_xgb = xgb_model.predict_proba(X_val)[:, 1]
    pred_xgb = (p_xgb >= 0.5).astype(int)
    rows.append({
        'stage': stage_name,
        'model': 'xgboost',
        'feature_count': len(features),
        'accuracy': float(accuracy_score(y_val, pred_xgb)),
        'roc_auc': float(roc_auc_score(y_val, p_xgb)),
    })

    return pd.DataFrame(rows)


## 4) Stage A: baseline features only

In [ ]:
baseline_results = evaluate_models(train, val, baseline_features, stage_name='baseline')
display(baseline_results)


## 5) Stage B: baseline + engineered features

In [ ]:
enhanced_results = evaluate_models(train, val, all_features, stage_name='enhanced')
display(enhanced_results)


## 6) Stage C: diagnostics (good features + bad features)

In [ ]:
diag = pd.read_csv(BASE / 'feature_diagnostics.csv')
keep_features = diag[diag['status'] == 'keep']['feature'].tolist()
drop_features = diag[diag['status'] == 'drop_candidate']['feature'].tolist()

print('Keep feature count:', len(keep_features))
print('Drop-candidate count:', len(drop_features))
display(diag[['feature','status','diagnostic_score','perm_auc_drop','goodness_score','xgb_gain','mean_abs_shap']].sort_values('diagnostic_score', ascending=False))

diag[['feature','status','diagnostic_score','perm_auc_drop','goodness_score','xgb_gain','mean_abs_shap']].to_csv(BASE / 'feature_diagnostics_phase_table.csv', index=False)


## 7) Stage D: rerun using reduced feature set (keep features)

In [ ]:
if len(keep_features) < 8:
    keep_features = diag.sort_values('diagnostic_score', ascending=False)['feature'].head(15).tolist()

reduced_results = evaluate_models(train, val, keep_features, stage_name='reduced_keep_set')
display(reduced_results)


## 8) Compare all stages and generate figures

In [ ]:
stage_results = pd.concat([baseline_results, enhanced_results, reduced_results], ignore_index=True)
stage_results = stage_results.sort_values(['model','stage'])
display(stage_results)
stage_results.to_csv(BASE / 'stage_model_comparison.csv', index=False)

plt.rcParams['figure.facecolor'] = BLACK
plt.rcParams['axes.facecolor'] = CARD
plt.rcParams['savefig.facecolor'] = BLACK

for metric in ['roc_auc', 'accuracy']:
    fig, ax = plt.subplots(figsize=(9, 5))
    pivot = stage_results.pivot(index='stage', columns='model', values=metric)
    pivot = pivot.reindex(['baseline', 'enhanced', 'reduced_keep_set'])
    x = np.arange(len(pivot.index))
    width = 0.35
    ax.bar(x - width/2, pivot['logistic'], width=width, color=YELLOW, label='logistic')
    ax.bar(x + width/2, pivot['xgboost'], width=width, color=BLUE, label='xgboost')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, color=WHITE)
    ax.set_title(f'Stage Comparison: {metric.upper()}', color=WHITE, fontsize=14, fontweight='bold')
    ax.set_ylabel(metric.upper(), color=WHITE)
    ax.tick_params(axis='y', colors=WHITE)
    ax.grid(axis='y', alpha=0.20, color=WHITE)
    for spine in ax.spines.values():
        spine.set_color(WHITE)
    leg = ax.legend(facecolor=CARD, edgecolor=WHITE)
    for t in leg.get_texts():
        t.set_color(WHITE)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'workflow_stage_comparison_{metric}.png', dpi=180)
    plt.show()
    plt.close(fig)

status_counts = diag['status'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
order = ['keep', 'review', 'drop_candidate']
vals = [int(status_counts.get(k, 0)) for k in order]
ax.bar(order, vals, color=[YELLOW, WHITE, BLUE])
ax.set_title('Feature Diagnostic Status Counts', color=WHITE, fontsize=13, fontweight='bold')
ax.set_ylabel('Count', color=WHITE)
ax.tick_params(axis='x', colors=WHITE)
ax.tick_params(axis='y', colors=WHITE)
ax.grid(axis='y', alpha=0.20, color=BLUE)
for spine in ax.spines.values():
    spine.set_color(WHITE)
fig.tight_layout()
fig.savefig(FIG_DIR / 'workflow_feature_status_counts.png', dpi=180)
plt.show()
plt.close(fig)
